# 🔬 Crystal Growth Data - 10-02-2025A

# 🔬 Germanium Crystal Growth — ML Analysis

Exploratory data analysis and predictive modeling on Germanium crystal growth data from the READS / Ge-STAR Lab.

| Column | Unit | Description |
|--------|------|-------------|
| **Power** | Watts | Heater power applied during growth |
| **Time** | Seconds | Elapsed time since start of growth run |
| **Growth Rate** | g/sec | Crystal mass growth rate |
| **Net Impurity Atoms** | count | Net impurity atoms introduced |
| **Impurity (Prev. Crystal)** | count | Net impurity from the previous crystal |
| **Detector Grade** | categorical | Detector grade region classification |

> **⚠️ Caution:** The dataset is small (~300 points across 10 runs). ML results are *exploratory* — not production-grade predictions.

In [ ]:
import warnings, io, glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold, GroupKFold, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

In [ ]:
# ─── Data: pulled live from GeDB ──────────────────────────────────────
# Dataset: Crystal Growth Data - 10-02-2025A
# Opened from GeDB → Run AI/ML. Rows come from the GeDB API (same data you see on the site).
import json, os, shutil, urllib.request
import pandas as pd

GEDB_API = "https://graphql.bicbioeng.org/graphql/gedb"
DATASET_ID = "crystal_growth_10-02-2025A"

_q = {"query": "query($t:String!,$e:String!,$g:String!){ nfbResult(target:$t, entity:$e, tag:$g, pageNumber:1, pageSize:25){ bulk } }",
      "variables": {"t": '"' + DATASET_ID + '"', "e": "datasets", "g": "gedb_datasets_mei_lab_datasets"}}
_req = urllib.request.Request(GEDB_API, json.dumps(_q).encode(), {"Content-Type": "application/json", "User-Agent": "curl/8 colab"})
_hits = [json.loads(h["bulk"]) for h in json.load(urllib.request.urlopen(_req, timeout=120))["data"]["nfbResult"]]
_rows = next(b["csv_file"] for b in _hits if b.get("id") == DATASET_ID and b.get("csv_file"))

shutil.rmtree("datasets", ignore_errors=True)   # only this dataset, even on re-run
os.makedirs("datasets")
_df = pd.DataFrame(_rows)
if "SheetName" in _df:   # merged file → one run per crystal, so per-run plots and CV work
    for sheet, part in _df.groupby("SheetName", sort=False):
        part.drop(columns="SheetName").to_csv(f"datasets/crystal_growth_{sheet.replace(' ', '_')}.csv", index=False)
else:
    _df.to_csv(f"datasets/crystal_growth_{DATASET_ID.replace('crystal_growth_', '')}.csv", index=False)
print(f"✅ {len(_df)} rows of '{DATASET_ID}' → datasets/")

CSV_URLS = []   # empty → the loader below reads datasets/crystal_growth_*.csv

POWER_OUTLIER_THRESHOLD = 30_000   # Watts — values above this are flagged
EXCLUDE_OUTLIERS_FROM_ML = True    # Set False to include outliers in ML training


In [ ]:
# ─── Data Loading & Header Normalization ──────────────────────────────
# Maps known header variations (lowercased) → canonical names.
# Extend this dictionary when new column name variations appear.

COLUMN_MAP = {
    'power_w':               ['power(w)', 'power (w)'],
    'time_sec':              ['time(sec)', 'time (sec)', 'total time(sec)', 'total time (sec)'],
    'growth_rate_g_per_sec': ['growth rate (g/sec)', 'growth rate(g/sec)', 'growth rate (gm/sec)', 'growth rate(gm/sec)'],
    'net_impurity_atoms':    ['no. of net impurity atoms added', 'no. of net impurity atoms introduced'],
    'impurity_prev_crystal': ['number of net impurity of previous crystal',
                              'no. of new impurity of previous crystal', 'number of net impurity of previous crystal added'],
    'detector_grade':        ['detector grade region', 'detector grade', 'detector grade portion (%)'],
}

_rev = {v: k for k, vs in COLUMN_MAP.items() for v in vs}
CANONICAL = list(COLUMN_MAP.keys())

def normalize_columns(df):
    renamed = {c: _rev.get(c.strip().lower(), c.strip().lower().replace(' ', '_')) for c in df.columns}
    df = df.rename(columns=renamed)
    available = [c for c in CANONICAL if c in df.columns]
    extra = [c for c in df.columns if c not in CANONICAL]
    return df[available + extra]

# Attempt to read CSV URLs from Colab query params
try:
    from google.colab import output
    from urllib.parse import parse_qs, urlparse
    import IPython
    url_str = IPython.display.Javascript('google.colab.kernel.proxyPort(0)')
    # Colab injects query params as environment-like variables
    # Researchers can pass ?csv=URL1&csv=URL2 in the Colab link
except Exception:
    pass

# Load data
frames = []

if CSV_URLS:
    for url in CSV_URLS:
        df = pd.read_csv(url)
        df = normalize_columns(df)
        name = url.split('/')[-1].replace('crystal_growth_', '').split('.csv')[0]
        df['source_file'] = name
        frames.append(df)
else:
    for fp in sorted(glob.glob('datasets/crystal_growth_*.csv')):
        df = pd.read_csv(fp)
        df = normalize_columns(df)
        df['source_file'] = os.path.basename(fp).replace('crystal_growth_', '').replace('.csv', '')
        frames.append(df)

if not frames:
    raise FileNotFoundError('No CSV data found. Populate CSV_URLS or place files in datasets/')

data = pd.concat(frames, ignore_index=True)
print(f'{len(frames)} files → {data.shape[0]} rows × {data.shape[1]} cols')
data.head()

In [ ]:
# ─── Outlier Detection (Flag, Don't Delete) ──────────────────────────
data['is_outlier'] = data['power_w'] > POWER_OUTLIER_THRESHOLD
n_out = data['is_outlier'].sum()
print(f'Outliers (Power > {POWER_OUTLIER_THRESHOLD:,} W): {n_out}/{len(data)} rows ({100*n_out/len(data):.1f}%)')

if n_out > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    clean, outliers = data[~data['is_outlier']], data[data['is_outlier']]
    for ax, col, label in zip(axes, ['power_w', 'growth_rate_g_per_sec'], ['Power (W)', 'Growth Rate (g/sec)']):
        ax.scatter(clean['time_sec'], clean[col], c='steelblue', alpha=.6, s=18, label='Normal')
        ax.scatter(outliers['time_sec'], outliers[col], c='red', s=55, marker='x', lw=2, label='Outlier')
        ax.set(xlabel='Time (sec)', ylabel=label, title=f'{label} — Outliers Highlighted')
        ax.legend()
    plt.tight_layout(); plt.show()
    display(data[data['is_outlier']])

# Handle NaNs
if data.isnull().any().any():
    data = data.dropna()
    print(f'Dropped NaN rows → {len(data)} remain')

In [ ]:
sources = data['source_file'].unique()
cmap = plt.get_cmap('tab10', len(sources))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Growth Rate vs Time
for i, src in enumerate(sources):
    s = data[data['source_file'] == src]
    axes[0].plot(s['time_sec'], s['growth_rate_g_per_sec'], marker='o', ms=3, lw=1.3, color=cmap(i), alpha=.8, label=src)
axes[0].set(xlabel='Time (sec)', ylabel='Growth Rate (g/sec)', title='Growth Rate Over Time')
axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)

# Power Profile
for i, src in enumerate(sources):
    s = data[data['source_file'] == src]
    axes[1].step(s['time_sec'], s['power_w'], where='post', lw=1.3, color=cmap(i), alpha=.8, label=src)
axes[1].set(xlabel='Time (sec)', ylabel='Power (W)', title='Power Profile Over Time')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)

plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution
sns.histplot(data['growth_rate_g_per_sec'], kde=True, bins=30, ax=axes[0], color='steelblue')
axes[0].set_title('Growth Rate Distribution')

# Box plot by Detector Grade
sns.boxplot(x='detector_grade', y='growth_rate_g_per_sec', data=data, ax=axes[1], palette='Set2')
axes[1].set_title('Growth Rate by Detector Grade')

# Correlation Heatmap
num_cols = [c for c in ['power_w','time_sec','growth_rate_g_per_sec','net_impurity_atoms','impurity_prev_crystal','detector_grade'] if c in data.columns]
corr = data[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=.5, ax=axes[2])
axes[2].set_title('Correlation Heatmap')

plt.tight_layout(); plt.show()

In [ ]:
# Power vs Growth Rate (scatter, colored by run — outliers excluded for readability)
plot_data = data[~data['is_outlier']]
fig, ax = plt.subplots(figsize=(10, 5))
for i, src in enumerate(sources):
    s = plot_data[plot_data['source_file'] == src]
    ax.scatter(s['power_w'], s['growth_rate_g_per_sec'], s=28, color=cmap(i), label=src, alpha=.7)
ax.set(xlabel='Power (W)', ylabel='Growth Rate (g/sec)', title='Power vs Growth Rate')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# Normalized time (fraction of total run)
data['time_normalized'] = data.groupby('source_file')['time_sec'].transform(lambda x: x / x.max() if x.max() > 0 else 0)

# Deltas within each crystal run
data['power_change'] = data.groupby('source_file')['power_w'].diff().fillna(0)
data['growth_rate_change'] = data.groupby('source_file')['growth_rate_g_per_sec'].diff().fillna(0)

# Log-transform impurity (sign-preserving)
for col in ['net_impurity_atoms', 'impurity_prev_crystal']:
    if col in data.columns:
        data[f'{col}_log'] = np.sign(data[col]) * np.log1p(np.abs(data[col]))

# Growth phase
data['growth_phase'] = pd.cut(data['time_normalized'], bins=[0, .25, .60, .85, 1.01],
                              labels=['early', 'mid', 'peak', 'decline'], include_lowest=True)

print(f'Features added. Shape: {data.shape}')
data[['source_file','time_normalized','power_change','growth_phase']].head(10)

In [ ]:
# Growth phases visualized
phase_colors = {'early': '#4CAF50', 'mid': '#2196F3', 'peak': '#FF9800', 'decline': '#F44336'}
fig, ax = plt.subplots(figsize=(10, 5))
for phase, color in phase_colors.items():
    s = data[data['growth_phase'] == phase]
    ax.scatter(s['time_normalized'], s['growth_rate_g_per_sec'], c=color, label=phase, alpha=.6, s=22)
ax.set(xlabel='Normalized Time', ylabel='Growth Rate (g/sec)', title='Growth Rate by Phase')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ─── Prepare ──────────────────────────────────────────────────────────
ml_data = data[~data['is_outlier']].copy() if EXCLUDE_OUTLIERS_FROM_ML else data.copy()

FEATURES = [c for c in ['power_w','time_normalized','detector_grade','power_change',
                         'net_impurity_atoms_log','impurity_prev_crystal_log'] if c in ml_data.columns]
TARGET = 'growth_rate_g_per_sec'

X, y = ml_data[FEATURES].values, ml_data[TARGET].values
groups = ml_data['source_file'].values

print(f'ML data: {len(ml_data)} rows | Features: {FEATURES}')

# ─── Train & Evaluate ─────────────────────────────────────────────────
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest':     RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42),
}

n_groups = len(np.unique(groups))
if n_groups >= 2:
    cv_strategy = GroupKFold(n_splits=min(10, n_groups))
    cv_kwargs = {'groups': groups}
    title = 'Leave-One-Crystal-Out CV'
else:
    cv_strategy = KFold(n_splits=min(10, max(2, len(X) // 5)), shuffle=True, random_state=42)
    cv_kwargs = {}
    title = 'Standard CV (Single Crystal)'

rows = []
for name, model in models.items():
    cv = cross_validate(model, X, y, cv=cv_strategy, **cv_kwargs,
                        scoring=['r2','neg_mean_absolute_error','neg_root_mean_squared_error'],
                        return_train_score=True)
    rows.append({
        'Model': name,
        'R² (train)': f"{cv['train_r2'].mean():.3f} ± {cv['train_r2'].std():.3f}",
        'R² (test)':  f"{cv['test_r2'].mean():.3f} ± {cv['test_r2'].std():.3f}",
        'MAE':  f"{-cv['test_neg_mean_absolute_error'].mean():.4f}",
        'RMSE': f"{-cv['test_neg_root_mean_squared_error'].mean():.4f}",
    })

print(f'\n── Model Comparison ({title}) ──')
display(pd.DataFrame(rows))

In [ ]:
# ─── Final Model & Visualizations ─────────────────────────────────────
rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42).fit(X, y)
gb = GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42).fit(X, y)
y_pred = rf.predict(X)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Actual vs Predicted
axes[0].scatter(y, y_pred, alpha=.5, s=22, c='steelblue', edgecolors='w', lw=.5)
lims = [min(y.min(), y_pred.min())-.02, max(y.max(), y_pred.max())+.02]
axes[0].plot(lims, lims, 'r--', lw=1.5)
axes[0].set(xlabel='Actual', ylabel='Predicted', title='Actual vs Predicted')

# Residuals
axes[1].scatter(y_pred, y - y_pred, alpha=.5, s=22, c='steelblue', edgecolors='w', lw=.5)
axes[1].axhline(0, color='r', ls='--', lw=1.5)
axes[1].set(xlabel='Predicted', ylabel='Residual', title='Residual Plot')

# Feature Importance (RF)
imp = pd.DataFrame({'Feature': FEATURES, 'Importance': rf.feature_importances_}).sort_values('Importance')
axes[2].barh(imp['Feature'], imp['Importance'], color='steelblue')
axes[2].set_title('Feature Importance (Random Forest)')

plt.tight_layout(); plt.show()

print(f'In-sample R²: {r2_score(y, y_pred):.3f} | MAE: {mean_absolute_error(y, y_pred):.4f}')

In [ ]:
# ─── Actual vs Predicted Growth Curves ────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
for i, src in enumerate(ml_data['source_file'].unique()):
    s = ml_data[ml_data['source_file'] == src]
    ax.plot(s['time_sec'], s[TARGET], marker='o', ms=3, lw=1, color=cmap(i), alpha=.6)
    ax.plot(s['time_sec'], rf.predict(s[FEATURES].values), lw=2, color=cmap(i), ls='--', alpha=.9, label=src)

ax.set(xlabel='Time (sec)', ylabel='Growth Rate (g/sec)', title='Actual (solid) vs Predicted (dashed)')
ax.legend(bbox_to_anchor=(1.02,1), loc='upper left', fontsize=7, title='Crystal Run')
plt.tight_layout(); plt.show()